# AI Toolkit by Ostris
## FLUX.1-dev Training


In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/ostris/ai-toolkit
!mkdir -p /content/dataset

Put your image dataset in the `/content/dataset` folder

In [ ]:
pip install sacremoses

In [ ]:
!cd ai-toolkit && git submodule update --init --recursive && pip install -r requirements.txt


## Model License
Training currently only works with FLUX.1-dev. Which means anything you train will inherit the non-commercial license. It is also a gated model, so you need to accept the license on HF before using it. Otherwise, this will fail. Here are the required steps to setup a license.

Sign into HF and accept the model access here [black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)

[Get a READ key from huggingface](https://huggingface.co/settings/tokens/new?) and place it in the next cell after running it.

In [ ]:
import getpass
import os

# Prompt for the token
hf_token = getpass.getpass('Enter your HF access token and press enter: ')

# Set the environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN environment variable has been set.")

In [ ]:
import os
import sys
sys.path.append('/content/ai-toolkit')
from toolkit.job import run_job
from collections import OrderedDict
from PIL import Image
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## Setup

This is your config. It is documented pretty well. Normally you would do this as a yaml file, but for colab, this will work. This will run as is without modification, but feel free to edit as you want.

In [ ]:
from toolkit.job import run_job  # Ensure this import path is correct

In [ ]:
from collections import OrderedDict
import torch
import json

# Detect device and set it appropriately
device = 'mps' if torch.backends.mps.is_built() else ('cuda:0' if torch.cuda.is_available() else 'cpu')

# Define job configuration
job_to_run = OrderedDict([
    ('job', 'extension'),
    ('config', OrderedDict([
        # Name of the job and configuration
        ('name', 'gearless_test'),
        ('process', [
            OrderedDict([
                ('type', 'sd_trainer'),
                # Folder paths for training and dataset
                ('training_folder', '/Users/mackcesar/PycharmProjects/ai-toolkit/output'),
                ('device', device),  # Dynamic device assignment
                ('network', OrderedDict([
                    ('type', 'lora'),
                    ('linear', 16),
                    ('linear_alpha', 16)
                ])),
                ('save', OrderedDict([
                    ('dtype', 'float32'),  # Use float32 for MPS compatibility
                    ('save_every', 250),  # Frequency for model checkpoint saving
                    ('max_step_saves_to_keep', 4)  # Number of checkpoints to retain
                ])),
                ('datasets', [
                    OrderedDict([
                        ('folder_path', '/Users/mackcesar/PycharmProjects/ai-toolkit/datasets/0b541f3d-b862-4e2a-9b38-1781274d9164'),
                        ('caption_ext', 'txt'),
                        ('caption_dropout_rate', 0.05),  # Caption dropout rate
                        ('shuffle_tokens', False),
                        ('cache_latents_to_disk', True),
                        ('resolution', [512, 768, 1024])  # Resolutions supported
                    ])
                ]),
                ('train', OrderedDict([
                    ('batch_size', 1),
                    ('steps', 2000),
                    ('gradient_accumulation_steps', 1),
                    ('train_unet', True),
                    ('train_text_encoder', False),
                    ('content_or_style', 'balanced'),
                    ('gradient_checkpointing', True),
                    ('noise_scheduler', 'flowmatch'),
                    ('optimizer', 'adamw'),
                    ('lr', 1e-4),
                    ('ema_config', OrderedDict([
                        ('use_ema', True),
                        ('ema_decay', 0.99)
                    ])),
                    ('dtype', 'float32')  # For MPS compatibility
                ])),
                ('model', OrderedDict([
                    ('name_or_path', 'black-forest-labs/FLUX.1-dev'),
                    ('is_flux', True),
                    ('quantize', False)  # Disable quantization for MPS compatibility
                ])),
                ('sample', OrderedDict([
                    ('sampler', 'flowmatch'),
                    ('sample_every', 250),
                    ('width', 1024),
                    ('height', 1024),
                    ('prompts', [
                        'woman with red hair, playing chess at the park, bomb going off in the background',
                        'a woman holding a coffee cup, in a beanie, sitting at a cafe',
                        'a horse is a DJ at a night club, fish eye lens, smoke machine, laser lights, holding a martini',
                        'a man showing off his cool new t-shirt at the beach, a shark jumping out of the water in the background',
                        'a bear building a log cabin in the snow-covered mountains'
                    ]),
                    ('neg', ''),
                    ('seed', 42),
                    ('walk_seed', True),
                    ('guidance_scale', 4),
                    ('sample_steps', 20)
                ]))
            ])
        ])
    ])),
    ('meta', OrderedDict([
        ('name', '[name]'),
        ('version', '1.0')
    ]))
])

# Function to print the job for verification
def print_job_config(job):
    #print("Job configuration:")
    print(json.dumps(job, indent=4))

# Print the job configuration to verify the structure
print_job_config(job_to_run)

# You can run the job with the following call:
# run_job(job_to_run)

## Run it

Below does all the magic. Check your folders to the left. Items will be in output/LoRA/your_name_v1 In the samples folder, there are preiodic sampled. This doesnt work great with colab. They will be in /content/output

In [ ]:
run_job(job_to_run)


In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from safetensors.torch import load_file
from diffusers import UNet2DModel  # Replace with the correct model class if needed
import os

# Detect device and set it appropriately
device = 'mps' if torch.backends.mps.is_built() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Done

Check your ourput dir and get your slider


In [ ]:
import torch
from safetensors.torch import load_file
from diffusers import UNet2DModel  # Ensure this is the correct model class

# Detect device and set it appropriately
device = 'mps' if torch.backends.mps.is_built() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to the locally saved .safetensors model file
model_path = "/Users/mackcesar/ComfyUI/models/unet/flux1-dev.safetensors"

# Load the state dictionary from the .safetensors file
try:
    state_dict = load_file(model_path)
    print(f"State dictionary loaded successfully from {model_path}")
except Exception as e:
    print(f"Failed to load state dictionary: {e}")
    raise

# Initialize the model architecture (ensure you use the correct model class)
try:
    # Adjust the parameters here as necessary for your model's architecture
    model = UNet2DModel(
        sample_size=64,  # Adjust as necessary
        in_channels=3,   # Adjust as necessary
        out_channels=3,
        layers_per_block=2,
        block_out_channels=(128, 256, 512, 1024)  # Example parameters, modify as needed
    )
    
    # Load the state dictionary with strict=False to avoid key mismatch issues
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    print(f"Model moved to {device} successfully.")
except TypeError as e:
    print(f"Error initializing model architecture: {e}")
    raise
except RuntimeError as e:
    print(f"Runtime error during model state loading: {e}")
    raise
except Exception as e:
    print(f"Unexpected error during model setup: {e}")
    raise

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, data_dir):
        self.image_files = [os.path.join(data_dir, file) for file in os.listdir(data_dir) if file.endswith(('.jpg', '.png'))]
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # Replace this with actual image loading and preprocessing logic
        image = torch.randn(3, 64, 64)  # Placeholder for an image tensor
        return image

data_dir = '/Users/mackcesar/PycharmProjects/ai-toolkit/datasets/0b541f3d-b862-4e2a-9b38-1781274d9164'
dataset = SimpleDataset(data_dir)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
# Set model to training mode
model.train()

# Define optimizer and loss function
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()  # Replace with appropriate loss function for your task

# Training loop
num_epochs = 5  # Adjust as needed
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    running_loss = 0.0
    for i, inputs in enumerate(dataloader):
        inputs = inputs.to(device)

        # Generate random timesteps for each batch
        batch_size = inputs.size(0)
        timesteps = torch.randint(0, 1000, (batch_size,), device=device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        try:
            output = model(inputs, timesteps)  # Include the `timesteps` argument

            # Check if the output has a 'sample' attribute (common in diffusion models)
            if hasattr(output, 'sample'):
                outputs = output.sample
            else:
                raise AttributeError("Model output does not contain 'sample' attribute")

            loss = criterion(outputs, inputs)  # Example: autoencoder-style training

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if (i + 1) % 10 == 0:  # Print every 10 batches
                print(f"Batch {i + 1}, Loss: {running_loss / (i + 1):.4f}")
        except AttributeError as e:
            print(f"AttributeError during forward pass: {e}")
            break
        except Exception as e:
            print(f"Unexpected error during forward pass: {e}")
            break

    print(f"Epoch {epoch + 1} completed. Average loss: {running_loss / len(dataloader):.4f}")

print("Training completed.")

In [ ]:
from safetensors.torch import save_file

# Define the path to save the model as a .safetensors file
save_path = "/Users/mackcesar/ComfyUI/models/unet/flux1-dev-trained.safetensors"

# Save the model's state_dict as a .safetensors file
save_file(model.state_dict(), save_path)
print(f"Model saved at {save_path}")

In [ ]:
import gc
import torch

def unload_model(model):
    """
    Unloads the model from memory and clears the cache.
    """
    if model is not None:
        del model  # Delete the model object
        gc.collect()  # Run garbage collection
        if torch.cuda.is_available():
            torch.cuda.empty_cache()  # Clear the GPU cache
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            torch.mps.empty_cache()  # Clear MPS cache (for Apple Silicon)

# Example usage: unload your model
unload_model(model)
print("Model unloaded and memory cleared.")